<h2 style=" color: #212618; font-weight:600;">NextHedge</h2>

<h2 style=" color: #212618; font-weight:600;">Implementation</h2>

For NextHedge, we opted for a class-based implementation for modularity outside of the notebook. The choice of leaving many of the variables configurable was driven by the fact that conditions significantly vary based on the industry, currency, and scenario. Thus, the configurations that come in our generation are usually observed values (suggested by AI) for the specific instance and will not be representative of every interaction in the treasury. We believe this flexibility is good for this use case.

The mathematical groundings and code implementation, which you can find both below, were assisted with artificial intelligence. Nonetheless, some choices like the GARCH model for spot rates were drawn from current econometric papers found online in simulating FX movements.

In [13]:
import sys
import os
sys.path.append(os.path.abspath("../scripts"))

import numpy as np
import pandas as pd
from scipy.stats import t as student_t
from datetime import datetime, timedelta

from data_qa import run_quality_check

In [6]:
class NextHedge:
    """
    Modular Treasury FX Risk Simulator
    -----------------------------------
    Simulates:
    - GARCH(1,1) FX spot rates with Student-t shocks
    - Smooth operational exposure paths
    - Policy-based hedge behavior
    - Probabilistic scenario injection
    - Derived treasury risk metrics
    """

    MATURITY_DAYS = {
        "1M":  30,
        "3M":  90,
        "6M": 180,
        "1Y": 365,
    }

    # Range (min, max) for sampling days_to_maturity per bucket
    MATURITY_DAYS_RANGE = {
        "1M":  (1,   30),
        "3M":  (30,  90),
        "6M":  (90, 180),
        "1Y":  (180, 365),
    }

    EXPOSURE_TYPES = ["transactional", "forecast", "translational"]

    def __init__(
        self,
        entities,
        currencies,
        maturities,
        base_currency="PHP",
        start_date="2026-01-01",
        days=252,
        config=None,
        entity_configs=None,
        currency_configs=None,
    ):
        self.entities = entities
        self.currencies = currencies
        self.maturities = maturities
        self.base_currency = base_currency
        self.start_date = datetime.strptime(start_date, "%Y-%m-%d")
        self.days = days

        default_config = {
            "omega": 1e-6,
            "alpha": 0.10,
            "beta": 0.85,
            "df": 6,
            "domestic_rate": 0.05,
            "foreign_rate": 0.03,

            "exposure_vol": 0.02,
            "exposure_lognorm_mean": 15.0,
            "exposure_lognorm_sigma": 0.5,

            "policy_hedge_ratio": 0.75,
            "hedge_deviation_vol": 0.05,
            "risk_limit_net_fc": 5_000_000,
            "hedge_rebalance_frequency": 30,
            "volatility_limit": 0.20,

            # Transaction arrival rate (Poisson lambda per day per grain)
            "tx_poisson_lam": 3,

            # Scenario probabilities: none, fx_shock, vol_shock, under_hedge
            "scenario_probs": [0.70, 0.10, 0.10, 0.10],
        }

        self.config = {**default_config, **(config or {})}
        self.entity_configs = entity_configs or {}
        self.currency_configs = currency_configs or {}
        self.drift_mu = self._compute_drift(self.config)

    # HELPERS
    @staticmethod
    def _compute_drift(cfg):
        return (cfg["domestic_rate"] - cfg["foreign_rate"]) / 252

    def _resolve_config(self, entity, currency=None):
        entity_override   = self.entity_configs.get(entity, {})
        currency_override = self.currency_configs.get(currency, {}) if currency else {}
        merged = {**self.config, **entity_override, **currency_override}
        merged["_drift_mu"] = self._compute_drift(merged)
        return merged

    @staticmethod
    def _maturity_to_days(maturity_bucket):
        return NextHedge.MATURITY_DAYS.get(maturity_bucket, 90)

    @staticmethod
    def _draw_scenario(probs):
        scenarios = ["none", "fx_shock", "vol_shock", "under_hedge"]
        return np.random.choice(scenarios, p=probs)

    # 1. FX SIMULATION (GARCH)
    def simulate_fx_path(self, s0, cfg, scenario):

        omega = cfg["omega"]
        alpha = cfg["alpha"]
        beta  = cfg["beta"]
        df    = cfg["df"]
        mu    = cfg["_drift_mu"]

        S        = np.zeros(self.days)
        sigma_sq = np.zeros(self.days)

        S[0]        = s0
        sigma_sq[0] = omega / (1 - alpha - beta)

        epsilon = student_t.rvs(df=df, size=self.days)

        vol_multiplier = 3.0 if scenario == "vol_shock" else 1.0
        fx_shock_day   = self.days // 2 if scenario == "fx_shock" else -1

        for t in range(1, self.days):

            log_return  = np.log(S[t-1] / S[t-2]) if t > 1 else 0.0
            residual    = log_return - mu
            sigma_sq[t] = omega + alpha * residual ** 2 + beta * sigma_sq[t - 1]
            sigma_t     = np.sqrt(sigma_sq[t] * vol_multiplier)

            S[t] = S[t - 1] * np.exp(mu - 0.5 * sigma_t ** 2 + sigma_t * epsilon[t])
            S[t] = max(S[t], 1e-10)

            if t == fx_shock_day:
                S[t] = max(S[t] * 0.85, 1e-10)

        return S, np.sqrt(sigma_sq)

    # 2. EXPOSURE SIMULATION
    def simulate_exposure_path(self, cfg, dates, maturity):
        """
        Generates transaction-level exposure rows using a Poisson arrival
        process. Each day, a random number of transactions arrive, each
        with an independently drawn lognormal amount. This reflects how
        real treasury exposure is composed of discrete invoices and contracts
        rather than a single aggregated daily figure.
        """

        lam   = cfg["tx_poisson_lam"]
        mu_ln = cfg["exposure_lognorm_mean"]
        sg_ln = cfg["exposure_lognorm_sigma"]

        lo, hi = self.MATURITY_DAYS_RANGE.get(maturity, (1, 90))

        rows = []
        for date in dates:
            n_tx = np.random.poisson(lam)
            if n_tx == 0:
                continue
            amounts = np.random.lognormal(mean=mu_ln, sigma=sg_ln, size=n_tx)
            for amt in amounts:
                rows.append({
                    "valuation_date":    date,
                    "gross_exposure_fc": round(amt, 2),
                    "exposure_type":     np.random.choice(self.EXPOSURE_TYPES),
                    "days_to_maturity":  np.random.randint(lo, hi + 1),
                })

        return pd.DataFrame(rows)

    # 3. HEDGE POLICY
    def simulate_hedge_ratio(self, cfg, scenario):

        policy        = 0.40 if scenario == "under_hedge" else cfg["policy_hedge_ratio"]
        deviation_vol = cfg["hedge_deviation_vol"]
        ratios        = np.clip(
            policy + np.random.normal(0, deviation_vol, self.days), 0, 1
        )

        return ratios

    # 4. FEATURE ENGINEERING
    def compute_metrics(self, df, cfg):

        df = df.sort_values(
            ["legal_entity", "currency", "maturity_bucket", "valuation_date"]
        ).reset_index(drop=True)

        df["hedge_notional_fc"] = df["gross_exposure_fc"] * df["hedge_ratio_actual"]
        df["net_exposure_fc"]   = df["gross_exposure_fc"] - df["hedge_notional_fc"]
        df["net_exposure_dc"]   = df["net_exposure_fc"] * df["spot_rate"]
        df["coverage_ratio"]    = df["hedge_notional_fc"] / df["gross_exposure_fc"]

        rd = cfg["domestic_rate"]
        rf = cfg["foreign_rate"]

        contract_rate_parts = []
        for (entity, ccy, mat), group in df.groupby(
            ["legal_entity", "currency", "maturity_bucket"]
        ):
            T = self._maturity_to_days(mat)
            F = group["spot_rate"] * np.exp((rd - rf) * T / 365)
            contract_rate_parts.append(F.shift(30).bfill())

        df["hedge_contract_rate"] = pd.concat(contract_rate_parts).reindex(df.index)
        df["mtm_hedge"]           = df["hedge_notional_fc"] * (df["spot_rate"] - df["hedge_contract_rate"])
        df["fx_delta"]            = df["net_exposure_fc"]
        df["VaR_95"]              = 1.65 * df["fx_volatility"] * np.abs(df["net_exposure_dc"])
        df["stress_loss_10pct"]   = df["net_exposure_fc"] * (-0.10 * df["spot_rate"])

        # Continuous maturity features
        df["tenor_weight"] = df["days_to_maturity"] / 365

        # Additional risk features
        df["vol_adjusted_exposure"]      = df["net_exposure_dc"] * df["fx_volatility"]
        df["duration_weighted_exposure"] = df["net_exposure_dc"] * df["tenor_weight"]
        df["fx_delta_1pct"]              = df["net_exposure_dc"] * 0.01

        df["hedge_ratio_policy"]        = cfg["policy_hedge_ratio"]
        df["risk_limit_net_fc"]         = cfg["risk_limit_net_fc"]
        df["base_currency"]             = self.base_currency
        df["hedge_rebalance_frequency"] = cfg["hedge_rebalance_frequency"]
        df["volatility_limit"]          = cfg["volatility_limit"]
        df["limit_breach"]              = df["net_exposure_fc"].abs() > df["risk_limit_net_fc"]

        return df

    # 5. MAIN GENERATOR
    def generate(self):

        all_data = []
        dates = [self.start_date + timedelta(days=i) for i in range(self.days)]

        starting_spots = {
            "USD": 55.0,
            "EUR": 60.0,
            "JPY":  0.38,
        }

        for entity in self.entities:
            for currency in self.currencies:

                cfg      = self._resolve_config(entity, currency)
                scenario = self._draw_scenario(cfg["scenario_probs"])

                s0     = starting_spots.get(currency, 50.0)
                S, vol = self.simulate_fx_path(s0, cfg, scenario)

                # Build a daily FX lookup for this currency path
                fx_lookup = pd.DataFrame({
                    "valuation_date": dates,
                    "spot_rate":      S,
                    "fx_volatility":  vol,
                })

                for maturity in self.maturities:

                    # Transaction-level exposure rows for this grain
                    tx_df = self.simulate_exposure_path(cfg, dates, maturity)

                    if tx_df.empty:
                        continue

                    # Merge FX path onto each transaction row by date
                    tx_df = tx_df.merge(fx_lookup, on="valuation_date", how="left")

                    # Draw one hedge ratio series for this grain, then sample
                    # a ratio per transaction by matching on valuation_date
                    hedge_ratios = self.simulate_hedge_ratio(cfg, scenario)
                    hr_lookup = pd.DataFrame({
                        "valuation_date":     dates,
                        "hedge_ratio_actual": hedge_ratios,
                    })
                    tx_df = tx_df.merge(hr_lookup, on="valuation_date", how="left")

                    tx_df["legal_entity"]    = entity
                    tx_df["currency"]        = currency
                    tx_df["maturity_bucket"] = maturity
                    tx_df["scenario"]        = scenario

                    all_data.append(tx_df)

        df = pd.concat(all_data, ignore_index=True)
        df = self.compute_metrics(df, self.config)

        return df

In [4]:
if __name__ == "__main__":

    entity_configs = {
        "TechCorp": {
            "policy_hedge_ratio":        0.85,
            "hedge_deviation_vol":       0.03,
            "risk_limit_net_fc":         3_000_000,
            "volatility_limit":          0.15,
            "hedge_rebalance_frequency": 14,
        },
        "AgriCo": {
            "policy_hedge_ratio":        0.60,
            "hedge_deviation_vol":       0.08,
            "risk_limit_net_fc":         7_000_000,
            "volatility_limit":          0.30,
            "hedge_rebalance_frequency": 30,
        },
    }

    currency_configs = {
        "USD": {
            "omega":        2e-7,
            "alpha":        0.08,
            "beta":         0.88,
            "df":           7,
            "foreign_rate": 0.05,
        },
        "EUR": {
            "omega":        5e-7,
            "alpha":        0.10,
            "beta":         0.85,
            "df":           6,
            "foreign_rate": 0.04,
        },
        "JPY": {
            "omega":                 1e-7,
            "alpha":                 0.06,
            "beta":                  0.90,
            "df":                    8,
            "foreign_rate":          0.001,
            "exposure_lognorm_mean":  19.0,
            "exposure_lognorm_sigma":  0.4,
        },
    }

    simulator = NextHedge(
        entities=["TechCorp", "AgriCo"],
        currencies=["USD", "EUR", "JPY"],
        maturities=["1M", "3M", "6M"],
        base_currency="PHP",
        config={"scenario_probs": [0.70, 0.10, 0.10, 0.10]},
        entity_configs=entity_configs,
        currency_configs=currency_configs,
    )

    df = simulator.generate()

    pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
    print(df.head(10).to_string())
    print(f"\nGenerated dataset shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nScenario distribution:\n{df.drop_duplicates(['legal_entity','currency'])['scenario'].value_counts()}")
    print(f"\nLimit breach rate: {df['limit_breach'].mean():.2%}")

  valuation_date  gross_exposure_fc  exposure_type  days_to_maturity  spot_rate  fx_volatility  hedge_ratio_actual legal_entity currency maturity_bucket   scenario  hedge_notional_fc  net_exposure_fc  net_exposure_dc  coverage_ratio  hedge_contract_rate     mtm_hedge       fx_delta         VaR_95  stress_loss_10pct  tenor_weight  vol_adjusted_exposure  duration_weighted_exposure  fx_delta_1pct  hedge_ratio_policy  risk_limit_net_fc base_currency  hedge_rebalance_frequency  volatility_limit  limit_breach
0     2026-01-01     4,485,669.2400  translational                27    60.0000         0.0032              0.5603       AgriCo      EUR              1M  vol_shock     2,513,240.6901   1,972,428.5499 118,345,712.9941          0.5603              60.0987 -248,085.1233 1,972,428.5499   617,499.3072   -11,834,571.2994        0.0740           374,242.0044              8,754,340.4133 1,183,457.1299              0.7500            5000000           PHP                         30            0.2

In [9]:
df.to_csv('../data/synthetic_treasury_risk_data.csv', index=False)

### Sanity Check

In [10]:
run_quality_check("../data/synthetic_treasury_risk_data.csv")

--- Running Treasury Data QA Audit ---
[PASS] Net Exposure logic is mathematically sound.
[PASS] No null values detected.


<h2 style=" color: #212618; font-weight:600;">Conceptual Modeling</h2>

In creation of the generator, these mathetmatical formulas serve as basis in modeling elements of treasury risk management.

### FX Market Behavior

**Spot Rate (GARCH):**

$$S_t = S_{t-1} \cdot \exp\!\left(\mu - \frac{\sigma_t^2}{2} + \sigma_t \cdot \varepsilon_t\right)$$

**Conditional Variance:**

$$\sigma_t^2 = \omega + \alpha \cdot \left(\log\frac{S_{t-1}}{S_{t-2}} - \mu\right)^2 + \beta \cdot \sigma_{t-1}^2$$

**Drift:**

$$\mu = \frac{r_d - r_f}{252}$$

**Noise term:**

$$\varepsilon_t \sim t(0, 1, v), \quad v \approx 5\text{--}8$$

**Stationarity constraint:**

$$\alpha + \beta < 1$$

**Unconditional variance (initialization):**

$$\sigma_0^2 = \frac{\omega}{1 - \alpha - \beta}$$

---

### Exposure Behavior

**Initial exposure:**

$$E_0^{FC} \sim \text{LogNormal}(\mu_E,\ \sigma_E)$$

**Exposure evolution:**

$$E_t^{FC} = E_{t-1}^{FC} \cdot (1 + \eta_t), \quad \eta_t \sim \mathcal{N}(0,\ \sigma_E^2)$$

---

### Hedge Behavior

**Hedge ratio:**

$$\text{HedgeRatio}_t = \theta + \xi_t, \quad \xi_t \sim \mathcal{N}(0,\ \sigma_\xi^2)$$

**Hedged amount:**

$$H_t^{FC} = \text{HedgeRatio}_t \cdot E_t^{FC}$$

**Hedge contract rate (Interest Rate Parity):**

$$S_{\text{contract}} = S_{t-30} \cdot \exp\!\left(\frac{(r_d - r_f) \cdot T}{365}\right)$$

**Hedge value:**

$$\text{HedgeValue}_t = H_t^{FC} \cdot S_{\text{contract}}$$

---

### Scenario Injections

**FX Shock:**

$$S_t^{\text{scenario}} = S_t \cdot (1 - 0.15)$$

**Volatility Shock:**

$$\sigma_t^{2,\text{scenario}} = 3 \cdot \sigma_t^2$$

**Under-hedging:**

$$\theta = 0.40$$

---

### Derived Risk Metrics

**Net exposure (FC):**

$$\text{NetExposure}_t^{FC} = E_t^{FC} - H_t^{FC}$$

**Net exposure (DC):**

$$\text{NetExposure}_t^{DC} = \text{NetExposure}_t^{FC} \cdot S_t$$

**Coverage ratio:**

$$\text{CoverageRatio}_t = \frac{H_t^{FC}}{E_t^{FC}}$$

**Mark-to-market PnL:**

$$\text{MTM}_t = H_t^{FC} \cdot (S_t - S_{\text{contract}})$$

**FX Delta:**

$$\Delta_t = \text{NetExposure}_t^{FC}$$

**Value at Risk (95%):**

$$\text{VaR}_{95} = 1.65 \cdot \sigma_t \cdot \left|\text{NetExposure}_t^{DC}\right|$$

**Stress Loss (10% depreciation):**

$$\text{StressLoss}_t = \text{NetExposure}_t^{FC} \cdot (-0.10 \cdot S_t)$$